# Multi-Model Comparison — Codebase Assistant

This evaluation notebook compares **multiple LLMs on the same repository and the same tasks**, using the existing Supervisor / agent pipelines.

**Important:** this notebook does not modify production agent logic. Each model is injected through a pinned OpenRouter client for a fair side-by-side run.

### Tasks compared
1. Code Analysis
2. Documentation Generation
3. Testing Agent

### Models
- Claude (via OpenRouter)
- Llama 3.1 8B Instruct
- Gemma 3 27B IT
- Nemotron Nano 9B V2 (when available)

Unavailable models are skipped gracefully with a recorded reason.

**Requirement:** `OPENROUTER_API_KEY` in `Project/.env`.

## 1. Setup

Add the project root to `sys.path`, import helpers, and confirm dependencies.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# Support running from Project/ or Project/notebooks/
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "codebase_assistant").is_dir() else CWD.parent
NOTEBOOKS = ROOT / "notebooks"
for path in (ROOT, NOTEBOOKS):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from codebase_assistant.config import Config
from compare_models_helpers import (
    COMPARISON_MODELS,
    markdown_table,
    probe_model,
    qualitative_discussion,
    resolve_repository,
    results_to_rows,
    run_model_comparison,
)

config = Config.load()
print("Project root:", ROOT)
print("OpenRouter key configured:", bool(config.openrouter_api_key))
print("Models in comparison set:", len(COMPARISON_MODELS))

## 2. Repository Selection

All models run on the **same** repository. Choose `demo` (default) or `medium`.

Do not change the repository between model runs in this notebook.

In [ ]:
# Keep this value fixed for the whole notebook run.
REPO_CHOICE = "demo"  # or "medium"

REPOSITORY = resolve_repository(REPO_CHOICE)
print("Using repository:", REPOSITORY)
print("Python files:")
for path in sorted(REPOSITORY.rglob("*.py")):
    print(" -", path.relative_to(REPOSITORY))

## 3. Model Availability Probe

Probe each model before the expensive agent runs. Unavailable models are skipped later with an explanation.

In [ ]:
availability = []
for display_name, slug in COMPARISON_MODELS.items():
    reason = probe_model(slug, config)
    availability.append(
        {
            "Model": display_name,
            "Slug": slug,
            "Available": reason is None,
            "Skip reason": reason or "",
        }
    )

availability_df = pd.DataFrame(availability)
display(availability_df)
display(
    Markdown(
        markdown_table(
            availability_df.to_dict(orient="records"),
            ["Model", "Slug", "Available", "Skip reason"],
        )
    )
)

## 4. Run Analysis / Documentation / Testing

For each available model the helper:

1. Builds an isolated Supervisor (separate Chroma / memory paths)
2. Pins a single OpenRouter model (no production fallback chain)
3. Runs **analysis → documentation → testing** on the shared repository
4. Collects runtime, tokens, grounded findings, docs length, tests passed, abstentions

This cell can take several minutes depending on indexing and provider latency.

In [ ]:
WORK_ROOT = NOTEBOOKS / ".compare_work"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

results = []
for display_name, slug in COMPARISON_MODELS.items():
    print("=" * 72)
    print(f"Running: {display_name} ({slug})")
    run = run_model_comparison(
        display_name,
        slug,
        REPOSITORY,
        work_root=WORK_ROOT,
    )
    results.append(run)
    if not run.available:
        print(f"Skipped: {run.skip_reason}")
    else:
        print(
            f"Done in {run.total_runtime_seconds:.2f}s | "
            f"grounded={run.grounded_findings} | "
            f"docs_len={run.documentation_length} | "
            f"tests_passed={run.tests_passed} | "
            f"abstentions={run.abstentions}"
        )
        if run.notes:
            print("Notes:", "; ".join(run.notes))

print("Completed model runs:", len(results))

## 5. Collect Metrics

Flatten per-model metrics into a DataFrame for tables and charts.

In [ ]:
rows = results_to_rows(results)
metrics_df = pd.DataFrame(rows)

available_df = metrics_df[metrics_df["Available"] == True].copy()
skipped_df = metrics_df[metrics_df["Available"] == False].copy()

display(Markdown("### Raw metrics"))
display(metrics_df)

if not skipped_df.empty:
    display(Markdown("### Skipped models"))
    display(skipped_df[["Model", "Slug", "Skip reason"]])

## 6. Comparison Tables

Markdown tables comparing quality, speed, and token usage.

In [ ]:
summary_columns = [
    "Model",
    "Runtime (s)",
    "Grounded findings",
    "Hallucinations rejected",
    "Docs length",
    "Test files",
    "Tests passed",
    "Abstentions",
]
token_columns = [
    "Model",
    "Prompt tokens",
    "Completion tokens",
    "Total tokens",
    "Latency (ms)",
    "Runtime (s)",
]

summary_rows = available_df.to_dict(orient="records") if not available_df.empty else rows

display(Markdown("### Quality & outcomes"))
display(Markdown(markdown_table(summary_rows, summary_columns)))

display(Markdown("### Speed & tokens"))
display(Markdown(markdown_table(summary_rows, token_columns)))

if not available_df.empty:
    display(Markdown("### Compact leaderboard"))
    compact = available_df[
        ["Model", "Runtime (s)", "Grounded findings", "Docs length", "Tests passed"]
    ]
    display(Markdown(markdown_table(compact.to_dict(orient="records"), list(compact.columns))))
else:
    display(Markdown("_No available models produced comparison rows._"))

## 7. Charts

Simple matplotlib comparisons for runtime, grounded findings, documentation length, and testing success.

In [ ]:
if available_df.empty:
    print("No available model results to chart.")
else:
    chart_df = available_df.set_index("Model")
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"Model comparison on {REPOSITORY.name}", fontsize=14)

    chart_df["Runtime (s)"].plot(kind="bar", ax=axes[0, 0], color="#3b6ea5")
    axes[0, 0].set_title("Runtime comparison")
    axes[0, 0].set_ylabel("Seconds")
    axes[0, 0].tick_params(axis="x", rotation=30)

    chart_df["Grounded findings"].plot(kind="bar", ax=axes[0, 1], color="#2a9d8f")
    axes[0, 1].set_title("Grounded findings")
    axes[0, 1].set_ylabel("Count")
    axes[0, 1].tick_params(axis="x", rotation=30)

    chart_df["Docs length"].plot(kind="bar", ax=axes[1, 0], color="#e9c46a")
    axes[1, 0].set_title("Documentation length")
    axes[1, 0].set_ylabel("Characters")
    axes[1, 0].tick_params(axis="x", rotation=30)

    chart_df["Tests passed"].plot(kind="bar", ax=axes[1, 1], color="#e76f51")
    axes[1, 1].set_title("Testing success (passed)")
    axes[1, 1].set_ylabel("Passed tests")
    axes[1, 1].tick_params(axis="x", rotation=30)

    fig.tight_layout()
    plt.show()

## 8. Qualitative Comparison / Discussion

Auto-generated from the metrics above. Edit the narrative cell if you want to add human judgment after reading the raw outputs.

In [ ]:
discussion = qualitative_discussion(results)
display(Markdown(discussion))

# Optional human notes after inspecting outputs:
human_notes = """
### Additional observations

- Review whether grounded findings cite real evidence from the shared repository.
- Prefer models that abstain instead of inventing documentation/tests when context is thin.
- Token totals are summed across analysis + documentation + testing calls for each model.
"""
display(Markdown(human_notes))

## 9. Conclusions

- Every model was evaluated on the **same repository** and the **same three agent tasks**.
- Production agent code was not modified; models were injected through evaluation helpers.
- Use the quality table for correctness signal (grounded findings / tests passed) and the token table for cost/speed trade-offs.
- Re-run with `REPO_CHOICE = "medium"` for a larger fixture without changing any other cell logic.

### How to re-run

```bash
jupyter notebook notebooks/Model_Comparison.ipynb
```

Ensure `OPENROUTER_API_KEY` is present in `Project/.env` before executing the run cell.